# Inspect a model

The examples below use `model` from
{doc}`tutorials/notebooks/11-model-building`, with `beta`, `variance`,
`sigma`, `mu`, `x`, and `y`. Inspect the model at its current values
before fitting it.

## Find inputs

In [1]:
import jax
import jax.numpy as jnp
import tensorflow_probability.substrates.jax.distributions as tfd

import liesel.model as lsl

In [2]:
# Python cell fragment: imports are visible in each guide.
x_data = jnp.linspace(-1.0, 1.0, 80)
y_data = 1.0 + 2.0 * x_data + 0.7 * jax.random.normal(jax.random.key(42), (80,))

prior_scale = lsl.Var.new_value(2.5, name="prior_scale")
beta = lsl.Var.new_param(
    jnp.zeros(2),
    dist=lsl.Dist(tfd.Normal, loc=0.0, scale=prior_scale),
    name="beta",
)
variance = lsl.Var.new_param(
    1.0,
    dist=lsl.Dist(tfd.LogNormal, loc=0.0, scale=0.5),
    name="variance",
)
x = lsl.Var.new_obs(x_data, name="x")

mu = lsl.Var.new_calc(
    lambda x, b: b[0] + b[1] * x,
    x,
    beta,
    name="mu",
)
sigma = lsl.Var.new_calc(jnp.sqrt, variance, name="sigma")
y = lsl.Var.new_obs(
    y_data,
    dist=lsl.Dist(tfd.Normal, loc=mu, scale=sigma),
    name="y",
)
model = lsl.Model(y)

# Match the tutorial after its reactive update.
beta.value = jnp.array([1.0, 2.0])


In [3]:
sorted(model.parameters)

['beta', 'variance']

In [4]:
sorted(model.observed)

['x', 'y']

In [5]:
response = model.vars["y"]

In [6]:
response.dist_node["scale"].name

'sigma'

In [7]:
model.vars["sigma"].value_node[0].name

'variance'

The parameters are `beta` and `variance`; the observed variables are `x`
and `y`. Their order in the mappings is not significant. The two input
lookups return `sigma` and `variance`. `"scale"` is the distribution's
argument name, whereas `"sigma"` is the variable's name. Use an integer for
an input passed positionally and its argument name for an input passed by keyword.

Strong variables receive their values directly; weak variables calculate them
from other variables. Observed and parameter flags describe a separate choice:
the statistical role of a variable. A weak variable can also carry a likelihood
or prior; see {doc}`model-distributions`.

## Read log densities

In [8]:
response.log_prob.shape

(80,)

In [9]:
{
    "likelihood": round(float(model.log_lik), 3),
    "prior": round(float(model.log_prior), 3),
    "total": round(float(model.log_prob), 3),
}

{'likelihood': -87.078, 'prior': -4.296, 'total': -91.374}

In [10]:
model.vars["x"].log_prob

0.0

The response contributes one log density per observation in this scalar normal
example. {attr}`model.log_lik <liesel.model.Model.log_lik>` sums densities on observed variables;
{attr}`model.log_prior <liesel.model.Model.log_prior>` sums densities on parameters. Here their sum is
{attr}`model.log_prob <liesel.model.Model.log_prob>`. Variables without a distribution, such as `x`, contribute
zero. A parameter without a distribution has a constant prior contribution;
on an unbounded domain this is an improper prior, with no distribution to sample.

The total normally sums all distribution nodes. A distribution not assigned
an observed or parameter role can therefore break the likelihood/prior
decomposition. Custom aggregate nodes can also change it. See
{doc}`model-distributions` before using such a model for optimization.

## Trace invalid values

Probe a copy so the original model keeps its valid values:

In [11]:
probe = model.copy()
probe.vars["variance"].value = -1.0
diagnostics = probe.diagnose().sort_values("name")

In [12]:
diagnostics[["name", "value_n_nan", "log_prob_n_nan", "log_prob_n_inf"]]

,name,value_n_nan,log_prob_n_nan,log_prob_n_inf
4,beta,0,0,0
5,mu,0,NaN,NaN
2,prior_scale,0,NaN,NaN
3,sigma,1,NaN,NaN
0,variance,0,1,0
1,x,0,NaN,NaN
6,y,0,80,0


In [13]:
probe.vars["variance"].value = 1.0

The negative variance makes its square root invalid. The table lets you trace
the NaN from `sigma` into the response density. Missing entries (`NaN`) mean that the
quantity does not apply, for example a log density on a variable without a
distribution. An infinite log density can instead signal a value outside a
distribution's support; check the distribution and its inputs rather than
replacing every non-finite result with a number.

Use {meth}`~liesel.model.Model.plot` to follow dependencies and
{meth}`~liesel.model.Model.plot_nodes` when you need to see the underlying value
and distribution nodes separately. Finite values establish neither a suitable
statistical model nor convergence of an inference algorithm.